# Week 6: Unsupervised Learning & Clustering
## Notebook 13: Mean Shift Non-Parametric Clustering from Scratch

### 1. Conceptual Intuition & Theoretical Background
Unlike K-Means (which requires pre-specifying $K$) or GMMs (which assume Gaussian shapes), **Mean Shift** is a **non-parametric density-gradient ascent** algorithm.

It treats the data points as samples drawn from an underlying empirical probability density function $f(x)$ estimated via **Kernel Density Estimation (KDE)**:
$$\hat{f}(x) = \frac{1}{n c_{k,d} h^d} \sum_{i=1}^{n} k\left( \left\| \frac{x - x_i}{h} \right\|^2 \right)$$

**Mean Shift Vector ($m(x)$):**
The gradient of density $\nabla f(x)$ points in the direction of maximum density increase:
$$m(x) = \frac{\sum_{i=1}^{n} x_i \cdot g\left( \left\| \frac{x - x_i}{h} \right\|^2 \right)}{\sum_{i=1}^{n} g\left( \left\| \frac{x - x_i}{h} \right\|^2 \right)} - x$$
Where $g(r) = -k'(r)$.
Points iteratively shift toward density modes: $x^{(t+1)} = x^{(t)} + m(x^{(t)})$. Points that converge to the same stationary mode are grouped into the same cluster.

---

### 2. Scratch Python Implementation: Mean Shift Clustering

In [ ]:
import numpy as np

class MeanShiftScratch:
    def __init__(self, bandwidth=1.5, max_iter=100, tol=1e-4):
        self.bandwidth = bandwidth
        self.max_iter = max_iter
        self.tol = tol
        self.cluster_centers = None
        self.labels = None
        
    def _flat_kernel(self, distance):
        # Flat kernel: 1 if inside bandwidth, 0 otherwise
        return np.where(distance <= self.bandwidth, 1.0, 0.0)
        
    def fit(self, X):
        n_samples, n_features = X.shape
        # Initialize candidate seeds at each data point
        shifted_points = X.copy()
        
        for i in range(n_samples):
            point = shifted_points[i]
            for _ in range(self.max_iter):
                # Compute distances to all points
                dists = np.linalg.norm(X - point, axis=1)
                weights = self._flat_kernel(dists)
                
                sum_weights = np.sum(weights)
                if sum_weights == 0:
                    break
                    
                # Compute new center of mass
                new_point = np.sum(X * weights[:, np.newaxis], axis=0) / sum_weights
                
                # Check convergence
                shift_dist = np.linalg.norm(new_point - point)
                point = new_point
                if shift_dist < self.tol:
                    break
                    
            shifted_points[i] = point
            
        # Group converged modes within bandwidth/2 into distinct cluster centers
        centers = []
        labels = np.zeros(n_samples, dtype=int)
        
        for idx, pt in enumerate(shifted_points):
            if not centers:
                centers.append(pt)
                labels[idx] = 0
            else:
                dists_to_centers = np.linalg.norm(centers - pt, axis=1)
                min_dist_idx = np.argmin(dists_to_centers)
                if dists_to_centers[min_dist_idx] < (self.bandwidth / 2.0):
                    labels[idx] = min_dist_idx
                else:
                    centers.append(pt)
                    labels[idx] = len(centers) - 1
                    
        self.cluster_centers = np.array(centers)
        self.labels = labels
        return self

# Benchmark on 3 Non-Concentric Clusters
np.random.seed(42)
c1 = np.random.normal(loc=[-4, -4], scale=0.5, size=(40, 2))
c2 = np.random.normal(loc=[0, 4], scale=0.5, size=(40, 2))
c3 = np.random.normal(loc=[4, -4], scale=0.5, size=(40, 2))
X_data = np.vstack([c1, c2, c3])

ms = MeanShiftScratch(bandwidth=2.0).fit(X_data)

print("--- Mean Shift Non-Parametric Clustering ---")
print(f"Total Samples:            {len(X_data)}")
print(f"Discovered Modes (K):     {len(ms.cluster_centers)} (Automatically found K=3!)")
print(f"Discovered Mode Centers:\n{np.round(ms.cluster_centers, 2)}")

### 3. Step-by-Step Code Walkthrough
1. **Density Gradient Ascent:** Shifts each candidate point to the weighted mean of its neighborhood.
2. **Automatic $K$ Discovery:** Clusters naturally emerge from stationary modes without specifying $K$.
3. **Mode Pruning:** Merges nearby stationary points into unified cluster centers.

---

### 4. Industry Applications & Trade-offs
- **Computer Vision Object Tracking:** Tracking moving objects in video streams via color histogram density ascent (CamShift).
- **Trade-off:** $O(N^2)$ distance computations per iteration make naive Mean Shift slow for large datasets.

---

### 5. Top Interview Questions & Answers
**Q1: How does Mean Shift determine the number of clusters $K$?**
> **Answer:** Mean Shift does not require $K$. It finds all local modes of the kernel density estimate. The number of discovered modes becomes the number of clusters, governed solely by the bandwidth parameter $h$.

**Q2: What is the impact of choosing too large or too small a bandwidth $h$?**
> **Answer:** If $h$ is too large, distinct modes smooth together into a single global cluster (under-segmentation). If $h$ is too small, noise creates many artificial local modes (over-segmentation).